In [3]:
with open('/content/the-verdict.txt',mode='r',encoding='utf-8') as file:
  raw_text = file.read()
print(raw_text)

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even through th

In [4]:
import tiktoken # byte pair encoding
tokenizer = tiktoken.get_encoding('cl100k_base')

In [5]:
encode_text = tokenizer.encode(raw_text)
print(encode_text)

[40, 473, 1846, 2744, 3463, 7762, 480, 285, 22464, 4856, 264, 12136, 35201, 313, 4636, 264, 1695, 12637, 3403, 313, 708, 433, 574, 912, 2294, 13051, 311, 757, 311, 6865, 430, 11, 304, 279, 2673, 315, 813, 27025, 11, 568, 1047, 12504, 813, 19354, 11, 12502, 264, 9257, 57896, 11, 323, 9749, 5678, 304, 264, 47625, 389, 279, 51768, 26919, 13, 320, 27831, 358, 4856, 3463, 433, 1053, 617, 1027, 22463, 477, 48606, 9456, 10227, 2673, 315, 813, 27025, 75857, 9210, 574, 1148, 279, 3278, 2663, 433, 13, 358, 649, 6865, 18083, 13, 480, 100242, 666, 24510, 313, 26301, 1566, 10780, 2503, 466, 313, 451, 501, 5620, 813, 653, 4711, 481, 671, 67, 20901, 13, 330, 2173, 3388, 433, 596, 2133, 311, 3708, 279, 907, 315, 856, 6945, 364, 3195, 709, 26, 719, 358, 1541, 956, 1781, 315, 430, 11, 4491, 13, 23194, 5721, 313, 1820, 4814, 311, 18925, 83, 374, 682, 358, 1781, 315, 1210, 578, 3492, 11, 389, 18083, 13, 666, 24510, 596, 23726, 11, 56016, 1202, 721, 5544, 62, 439, 3582, 814, 1051, 27000, 304, 459, 26762, 4

In [6]:
print(len(encode_text))

4943


Executing the code above will return 4943, the total number of tokens in the training set, after applying the BPE tokenizer.

Next, we remove the first 50 tokens from the dataset for demonstration purposes as it results in a slightly more interesting text passage in the next steps

In [7]:
enc_sample = encode_text[50:]
print(len(enc_sample))

4893


One of the easiest and most intuitive ways to create the input-target pairs for the nextword prediction task is to create two variables, x and y, where x contains the input tokens and y contains the targets, which are the inputs shifted by 1 like stride

The context size determines how many tokens are included in the input

# Here we have created input-target pair
like x is input_label and y is the target_label

In [8]:
context_size = 4 #length of the input
#The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens)
#to predict the next word in the sequence.
#The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

x = enc_sample[:context_size] # the is input_text
y = enc_sample[1:context_size+1]# this is target(output) which input trainig after prediction
print(f'x{x}') #here stride is 1
print(f'y      {y}')

x[323, 9749, 5678, 304]
y      [9749, 5678, 304, 264]


In [9]:
for i in range(1, context_size+1): # context_size = 4 +1 = 5
    context = enc_sample[:i]#we have define the enc_sample so 1 index from the enc_sample
    desired = enc_sample[i]

    print(context, "---->", desired)

[323] ----> 9749
[323, 9749] ----> 5678
[323, 9749, 5678] ----> 304
[323, 9749, 5678, 304] ----> 264


In [10]:
from IPython.utils.py3compat import decode
for i in range(1, context_size+1): # context_size = 4 +1 = 5
    context = enc_sample[:i]#we have define the enc_sample so 1 index from the enc_sample
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

#context is already a list of token IDs, so you can decode it directly.
#desired is a single integer, so you wrap it in [] to create a list containing that ID.

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


2. Dataset

Creates or retrieves input-target pairs

Input: [i, love, the]

Target: [love, the, machine]

3. DataLoader

Groups multiple pairs into batches , input-target pairs

# IMPLEMENTING A DATA LOADER
This class prepares text data for training a GPT model. It converts text into token IDs, creates input-target pairs using a sliding window, and makes those pairs available to PyTorch's DataLoader

#Creates and stores input-target pairs from tokenized text

In [13]:

import torch
from torch.utils.data import Dataset, DataLoader


# Create a custom Dataset class for GPT training.
# Dataset helps us organize and retrieve input-target pairs.
class GPTDatasetV1(Dataset):

    # Initialize the dataset with text, tokenizer, context length, and stride.
    def __init__(self, txt, tokenizer, max_length, stride):

        # Create empty lists to store input sequences (X) and target sequences (Y).
        self.input_ids = []
        self.target_ids = []

        # Convert the entire text into a sequence of token IDs.
        # allowed_special allows the tokenizer to recognize <|endoftext|>
        # as a special token if it appears in the text.
        token_ids = tokenizer.encode(
            txt,
            allowed_special={"<|endoftext|>"}
        )

        # Move a sliding window through the token IDs.
        # max_length determines the number of tokens in each input.
        # stride determines how far the window moves each time.
        # A smaller stride creates more overlapping examples.
        for i in range(0, len(token_ids) - max_length, stride):

            # Select max_length tokens starting at position i.
            # These tokens are the input (X) given to the model.
            input_chunk = token_ids[i:i + max_length]

            # Select the next max_length tokens, shifted one position forward.
            # These tokens are the target (Y) the model should learn to predict.
            target_chunk = token_ids[i + 1:i + max_length + 1]

            # Convert the input list into a PyTorch tensor and store it.
            self.input_ids.append(torch.tensor(input_chunk))

            # Convert the target list into a PyTorch tensor and store it.
            self.target_ids.append(torch.tensor(target_chunk))

    # Return the total number of input-target pairs in the dataset.
    # PyTorch uses this to determine how many examples are available.
    def __len__(self):
        return len(self.input_ids)

    # Retrieve one input-target pair using its index (idx).
    # For example, dataset[0] returns the first input and its target.
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


#Creates a Dataset and wraps it in a DataLoader

# DataLoader

In [14]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [15]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

#Calls the function and stores the resulting DataLoader

In [19]:

import torch

# Display the installed PyTorch version.
print("PyTorch version:", torch.__version__)

# Create a DataLoader that supplies batches from the raw text.
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,   # Get 1 training example per batch.
    max_length=4,   # Each input contains 4 tokens.
    stride=1,       # Move the sliding window forward by 1 token.
    shuffle=False   # Keep examples in their original order.
)

# Create an iterator that retrieves batches one at a time.
data_iter = iter(dataloader)

# Retrieve the first batch from the iterator.
first_batch = next(data_iter)

# Display the first batch, containing inputs and targets.
print(first_batch)


PyTorch version: 2.11.0+cpu
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [17]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [18]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
